# CryoCast Pipeline Demo

This demonstration showcases the complete CryoCast ML pipeline capabilities through CLI commands.

**Target Audience:** Developer teams and future team members who want to understand our design decisions, 
trade-offs, and flexible experimentation capabilities.

**You'll learn how to:**
- Run our training pipeline end-to-end in three lines of code
- Swap between different modelling paradigms
- Reproduce runs and inspect the outputs

<div class="alert alert-danger">

If you want to run this notebook, you will need a **CDS account** in order to download the ERA5 weather data. Details of how to set this up can be found [here](https://cds.climate.copernicus.eu/how-to-api).

You will also need a **Weights & Biases account** to log the training runs. You can sign up for a free account [here](https://wandb.ai/). Once you have an account, you will need to set up your API key; instructions for doing so can be found [here](https://docs.wandb.ai/models/quickstart#python-notebook).

</div>

## Notebook Structure

[**Section 1: End-to-End Training**](#section-1-end-to-end-training-pipeline)
- Run a full CryoCast pipeline end-to-end using a minimal configuration & data
- Inspect training artifacts and see evaluation outputs

[**Section 2: Model Flexibility**](#section-2-model-flexibility)
- Switch between Encode-Process-Decode paradigm and standalone persistence model
- Explore Encoder module functionality (Multimodality)

[**Section 3: Full flexibility - Advanced Example**](#section-3-full-flexibility-advanced-example)
- Write or adapt config files to change pipeline behavior
- Use anemoi functionality to fetch and inspect standard datsets

# Section 1: End-to-End Training Pipeline

In this section, we'll demonstrate the complete training pipeline using a simple **U-Net model with a naive encoder / decoder** (more details of this can be found below in [section 2](#section-2-model-flexibility)). 
The dataset contains sea ice concentration data (OSISAF) and corresponding atmospheric data (ERA5).
We don't expect the model to do well as we are only training for 10 epochs, and won't do any hyperparameter optimisation. However, it will give us a sense of the pipeline.

<div class="alert alert-danger"> 

N.B. to run this example requires you to have a **Weights & Biases account** to log the training runs. You can sign up for a free account [here](https://wandb.ai/). Once you have an account, you will need to set up your API key; instructions for doing so can be found [here](https://docs.wandb.ai/models/quickstart#python-notebook).

</div>


You can install the repo by running the following commands in your terminal:

```bash
git clone https://github.com/alan-turing-institute/cryocast
cd cryocast
pip install .
```

In [ ]:
# load in the relevant packages
from pathlib import Path
from IPython.display import Image, Video

# log in to wandb
!wandb login 

### Environment Verification

Let's verify that our cryocast CLI tools are available and working.

To run this notebook, you'll need a kernel (e.g. conda or .venv) with `cryocast` and `jupyter` installed.

In [ ]:
!cryocast --help

### Download the dataset for running the model

<div class="alert alert-danger"> 

To run this minimal example requires you to have signed up to access the ERA5 data. Details on how to do so can be found [here](https://cds.climate.copernicus.eu/how-to-api).

</div>

The model will download some sample data for the southern hemisphere. The datasets to download are specified in `demo_notebook.yaml` and consist of:

- weather data (ERA5)
- sea ice data (OSISAF)
- ocean data (Argo)

for a set of dates between 2020 and 2024.

(_Details about how these config files work will be covered in [section 3](#section-3-full-flexibility-advanced-example)._)

If the data is already present, a summary of the dataset will be printed (it will not be downloaded again). This same summary can be created by running `cryocast datasets inspect`.

**This downloads the data to a folder called `my_data/` in the root of the repo.**

In [ ]:
!cryocast datasets create --config-name=demo_notebook.yaml

### Train the model

The next command will train a simple U-Net model for sea ice concentration forecasting, and the training will run for 2 epochs. 

In [ ]:
# train the UNet model
!cryocast train --config-name=demo_notebook.yaml

# get the path to the checkpoint file
ckpt_unet_train = sorted(Path("../my_data/training/wandb/latest-run/checkpoints").glob("epoch=*-step=*.ckpt"))[-1].resolve()

### Evaluate the model

Finally the model is evaluated using the checkpoint, saved during the training run. The results are then logged to Weights & Biases. (This will log them to your default WandB project. Within the Sea Ice project there is a WandB account for the Sea Ice project where results are usually logged.)

In [ ]:
# evaluate the UNet model
!cryocast evaluate --config-name=demo_notebook.yaml --checkpoint="{ckpt_unet_train}"

# get the path to the media folder
media_unet_eval = Path("../my_data/training/wandb/latest-run/files/media").resolve()

We can then check the predictions from this model. (_N.B. as it is only run for a small number of epochs on a limited dataset, the results are not great._)

In [ ]:
# compare ground truth to the UNet prediction
video_unet = sorted((media_unet_eval / "videos" / "output_video").glob("*truth-vs-prediction_*"))[-1]
Video(video_unet, embed=True, width=800)

# Section 2: Model Flexibility

In this section, we'll demonstrate how easy it is to switch between different model architectures.
We'll also show the difference between standalone models and processor models.

The conceptually simpler model type is a standalone model, which takes in the input data and directly outputs the prediction. These models are less flexible, as they have to be specifically coded to handle new input / output data. Consequently a separate instance of the model is likely to be needed for each input / output combination. However, the input variables are available without transformation.  

![pipeline_standalone](../docs/src/assets/pipeline-standalone.png)

The more complex model type is processor model, which uses an encode-process-decode paradigm. Here, the input data is first encoded into a latent representation of fixed dimensions, which is then processed by a core model, before being decoded back into the output space. This allows for more flexibility in terms of input and output variables, as well as the ability to use different types of models for each component.

![pipeline_encode_process_decode](../docs/src/assets/pipeline-encode-process-decode.png)

## Using an alternative processor model

The initial model we ran, the `naive_unet_naive` model, is an example of a processor model. It uses a naive encoder, to convert the input variables into the latent space. A U-Net model is then run on the latent space, before a naive encoder extracts the SIC predictions. 

In this example, we switch the `naive_unet_naive` model for the alternative `cnn_vit_cnn` model, which uses a cnn enocder / decoder, and replaces the U-Net model with a Vision Transformer (ViT).

In [ ]:
# train the ViT model
!cryocast train --config-name=demo_notebook.yaml model=cnn_vit_cnn 

# get the path to the checkpoint file
ckpt_vit_train = sorted(Path("../my_data/training/wandb/latest-run/checkpoints").glob("epoch=*-step=*.ckpt"))[-1].resolve()

In [ ]:
# evaluate the ViT model
!cryocast evaluate --config-name=demo_notebook.yaml model=cnn_vit_cnn --checkpoint="{ckpt_vit_train}" 

# get the path to the media folder
media_vit_eval = Path("../my_data/training/wandb/latest-run/files/media").resolve()

The results are logged to [Weights & Biases](https://wandb.ai/), but they can also be viewed locally. When inspecting the predictions of the model using a vision transformer architecture you can notice a checkerboard pattern, an artefact of the patch embedding approach this model uses, which occurs when the models are not sufficiently trained.

In [ ]:
# compare ground truth to the ViT prediction
video_vit = sorted((media_vit_eval / "videos" / "output_video").glob("*truth-vs-prediction_*"))[-1]
Video(video_vit, embed=True, width=800)


There is also a set of other possible model options, however these struggle to train on a laptop, so we won't demonstrate them here. If you want to test them out, see the full list under `config/model` and set the model to your preferred option.

## Standalone persistence model

As mentioned above, an alternative form of model doesn't use an encoder / decoder architecture. We can demonstrate this with a `persistence model` which simply outputs the last input frame as the prediction.

In this case, as well as specifying the model type as `persistence`, we also need to specify a different training config, as the persistence model doesn't need training over multiple epochs.

In [ ]:
# train the persistence model
!cryocast train --config-name=demo_notebook.yaml train=persistence model=persistence

# get the path to the checkpoint file
ckpt_persistence_train = sorted(Path("../my_data/training/wandb/latest-run/checkpoints").glob("epoch=*-step=*.ckpt"))[-1].resolve()

In [ ]:
# evaluate the persistence model 
!cryocast evaluate --config-name=demo_notebook.yaml train=persistence model=persistence --checkpoint="{ckpt_persistence_train}"

# get the path to the media folder
media_persistence_eval = Path("../my_data/training/wandb/latest-run/files/media").resolve()

In [ ]:
# compare ground truth to the persistence prediction
video_persistence = sorted((media_persistence_eval / "videos" / "output_video").glob("*truth-vs-prediction_*"))[-1]
Video(video_persistence, embed=True, width=800)

# Section 3: Full flexibility (Advanced Example)

This section shows how to adapt the different parts of the model pipeline to your needs. We will look in to the config files that are the basis of the pipeline, and we will show how to create your own config file to give you full control over the model training. 

The pipeline uses hydra, which is a powerful configuration management tool. More details can be found [here](https://hydra.cc/docs/intro/). 

These config files are logged as part of the WandB run, so it is easy to go back and check exactly what settings were used for each run, or to reproduce a run.

## The base config file

So far in this notebook, we've mostly used the `demo_notebook.yaml` config file. We can have a look at the contents of this file to see what options it is using.

In [ ]:
!cat ../cryocast/config/demo_notebook.yaml

As you can see here, the defaults section points to the `base.yaml` config file, which contains the main options for the pipeline. There are then some specific options for how many epochs to train for, and where to save the data.

So lets have a look at the `base.yaml` file.

In [ ]:
!cat ../cryocast/config/base.yaml

There are lots more options under the defaults section here, for specifying a range of options such as the datasets, or the model to use. Each of these is another config file, in the folder structure. Some of these options should also look familiar from the `demo_notebook.yaml` file. For example, under `loggers`, we can see `wandb` specified as the logger to use.

We can explore the full list of config folders / files in the `cryocast/config/` folder, to get an idea of the options available. 

In [ ]:
!ls ../cryocast/config/*/

Looking at the contents, you can see that all the defaults in the base config, map to specific folders / files. Each file contains a different set of options that are being used for different parts of the pipeline. You can also see the alternative files that could be substituted in (e.g. using `cnn_vit_cnn.yaml` instead of naive_unet_naive, which is the `quick_test.yaml` for the model).

If you only want to change a single parameter, rather than the whole config file, you can specify this in the config, referencing the nested structure. An example of this is how we specified the number of epochs to train for in the `demo_notebook.yaml` file. Here, we overrode the default of 50 epochs specified in `train/trainer/default.yaml`, by setting `max_epochs` to be 2.

In this file list, you can also see a set of alternative root config files, which we'll look in to in the next section.

### Alternative root config files

Let's look at some of the root config files available in the `configs/` folder. 

In [ ]:
!ls ../cryocast/config/*.yaml

As you can see, the `demo_notebook.yaml` file we've been using is listed here. 

The list of baseline models that we use for testing can be found in the `baseline` folder:

In [ ]:
!ls ../cryocast/config/baseline/*.yaml

We can see a file called `persistence.yaml`.

In [ ]:
!cat ../cryocast/config/baseline/persistence.yaml

Here you can see that it mostly uses the same parameters as the `base` config. However, it overrides the default configs for `train` and `model` to switch in the persistence model options. 

When we ran the persistence model above, we specified these two `override` settings in the command line. An easier way to run the `persistence` model is to use the `persistence.yaml` config file. However, to run this on your local machine you will need to specify at least the file path to the data folder (the default path is set up for running on a cluster). This could be done with the command:

```bash
cryocast train --config-name persistence.yaml +data.path=./my_data/
```

<div class="alert alert-success">

Typically you should not change the values in the config files. If you want to change multiple of these options, the best way to do so is to create your own config file, similar to `demo_notebook.yaml`, It's easiest to use the `base` config as the default, and then specify the parameters you want to change using `override` or by specifying the parameter in the config hierarchy. 

When you want to run code using that config file, you can specify it using the `--config-name` option in the CLI command. This will remove the need to specify the individual parameters in the command line each time. 

</div>

## Creating a dataset

The first step in the model pipeline is to download the required input data. This has been developed to make use of the [anemoi datasets](https://anemoi.readthedocs.io/projects/datasets/en/latest/) package, which is part of the [anemoi toolkit](https://anemoi.readthedocs.io/en/latest/) developed by ECMWF. 

New datasets can be downloaded using the `cryocast datasets create` command shown in the first section of this notebook. 

If the data has already be downloaded, the `cryocast datasets inspect` command prints a summary of the dataset (or datsets), including a full list of the variables and some summary statistics. 

In [ ]:
!cryocast datasets inspect --config-name=demo_notebook --verbose

The details of the dataset to be downloaded are defined in the datasets config file. The default datasets (those used by the `base` config) are `samp_sicsouth_osisaf_25p0km_2020_2024_24h_v1`, `samp_weathersouth_era5_25p0km_2020_2024_24h_v4` and `samp_floatsouth_argo_25p0km_2020_2024_24h_v3`. We will have a look at these in a bit more detail.

In [ ]:
!cat ../cryocast/config/data/datasets/samp_sicsouth_osisaf_25p0km_2020_2024_24h_v1.yaml

This dataset contains sea ice concentration (SIC) data from the OSI SAF 25km dataset for the years 2020-2024, with a temporal resolution of 24 hours. The data is from the Southern Hemisphere. This download makes use of data preprocessing developed as part of the [IceNet repository](https://github.com/icenet-ai/icenet).

In [ ]:
!cat ../cryocast/config/data/datasets/samp_weathersouth_era5_25p0km_2020_2024_24h_v4.yaml

This dataset contains 50km resolution weather data from ERA5 for the years 2020-2024, with a temporal resolution of 24 hours. The data is from the Southern Hemisphere. The variables downloaded are those commonly used in sea ice forecasting, including temperature, u and v wind components, surface / sea level pressure, humidity and geopotential height. It also includes the cos and sin julian day to help the model learn seasonal patterns.

_N.B. there are also a set of other configs in the `config/datasets` folder for different subsets of variables, different hemispheres, and different datasets._

## Training a model

Training a model is done using the `cryocast train` command, as shown in [section 1](#train-the-model) of this notebook. There are a set of config files that specifically relate to the model training. For example the choice of model, the training parameters, the variable to predict, the dataset split and the logger to use. We will explore each of these in turn.

### Model choice

In [ ]:
!cat ../cryocast/config/model/quick_test.yaml

This config points to the specific python code used for each part of the model (i.e. the naive encoder, the U-Net processor and the naive decoder). There are various parameters that can be set, such as the size of the latent space and the number of start-out channels for the U-Net. 

There are a set of possible model configs - each of these will have specific parameters that need to be set for those models, and their default values. 

### Training parameters

In [ ]:
!cat ../cryocast/config/train/default.yaml

A set of training parameters are specified here, normally it is fine to just use the default values. 

### Prediction window parameters

In [ ]:
!cat ../cryocast/config/window/forecast-3-history-2.yaml

The `variables` config key specifies the variable(s) to be predicted, while the `window` config key specifies the number of historic days to include as input, and the number of days to forecast ahead. There are config files for predicting SIC in the northern or the southern hemisphere.

### Dataset split

In [ ]:
!cat ../cryocast/config/data/split/sample_dataset.yaml

The dataset split specifies the number of batches to use for training. It also specifies how the datasets should be split in to training, validation and test sets, based on date ranges. There are different splits for the sample or the full dataset, however the dates shouldn't be altered, to ensure that all the different model runs are comparable.

### Loggers

In [ ]:
!cat ../cryocast/config/reporting/loggers/wandb.yaml

By default, the pipeline uses Weights & Biases to log the training and evaluation metrics. All runs are saved to the 'turing-seaice' project, as well as being saved locally in a `training` folder alongside the data. We don't log the model weights to WandB to save space.

## Evaluating a model

Training a model produces a set of artifacts, including one or more model checkpoint. The checkpoints can be used to evaluate the model performance using the `cryocast evaluate` command (as shown in [section 1](#evaluate-the-model) of this notebook). There are a couple of config files that specifically relate to the evaluation process within the `evaluate` folder (though several of the ones we have already explored are also relevant).

### Plotting

In [ ]:
!cat ../cryocast/config/evaluate/callbacks/media_logging.yaml

By default, static maps and videos of the predicted forecasts of the output variable (normally SIC) are created for a set of examples from the test set. These are uploaded to Weights & Biases, and also saved locally. 

There are a lot of options for adjusting the model output plots. 

### Metric summary

In [ ]:
!cat ../cryocast/config/evaluate/callbacks/metric_summary.yaml

The default metric summary is based on average loss.

<div class="alert alert-success">

Typically you should not change the values in the config files. If you want to change multiple of these options, the best way to do so is to create your own config file, similar to `demo_notebook.yaml`, It's easiest to use the `base` config as the default, and then specify the parameters you want to change using `override` or by specifying the parameter in the config hierarchy. 

When you want to run code using that config file, you can specify it using the `--config-name` option in the CLI command. This will remove the need to specify the individual parameters in the command line each time. 

</div>

Hopefully this notebook has given you an overview of the CryoCast forecasting pipeline, and shown you how you can use it flexibily to run different models and evaluate their performance.